In [1]:
import os
import sys

# Ép PySpark sử dụng Python hiện tại và bỏ qua các kiểm tra Unix không cần thiết
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [2]:
import findspark
findspark.init()


In [3]:
from pyspark.sql import SparkSession

In [4]:
!pip install pyspark time-uuid


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
%pip install cassandra-driver

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from cassandra.util import datetime_from_uuid1
from cassandra.cqltypes import TimeUUIDType
import uuid 
from uuid import UUID
import time_uuid 
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf, col
import pyspark.sql.functions as sf

In [7]:
spark = SparkSession.builder \
    .appName("ETL_Testing") \
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1,mysql:mysql-connector-java:8.0.28") \
    .config("spark.cassandra.connection.host", "localhost") \
    .config("spark.cassandra.connection.port", "9042") \
    .getOrCreate()

In [8]:
data = spark.read.format("org.apache.spark.sql.cassandra") \
    .options(table='tracking', keyspace='keyspace_name') \
    .load()


In [9]:
def process_df(df):
    # 1. Định nghĩa UDF để chuyển đổi TimeUUID
    @udf(returnType=StringType())
    def to_datetime_str(x):
        if x is None: 
            return None
        # Chuyển đổi trực tiếp từ string/bytes của TimeUUID sang datetime string
        return time_uuid.TimeUUID(bytes=UUID(x).bytes).get_datetime().strftime('%Y-%m-%d %H:%M:%S')

    # 2. Dùng .withColumn để tạo cột 'ts' trực tiếp trên toàn bộ cụm (Cluster)
    # Không dùng .collect(), dữ liệu vẫn nằm trên các máy con để xử lý song song
    result = df.withColumn('ts', to_datetime_str(col('create_time'))) \
               .select(
                   'create_time', 
                   'ts', 
                   'job_id', 
                   'custom_track', 
                   'bid', 
                   'campaign_id', 
                   'group_id', 
                   'publisher_id'
               )
    
    return result

In [10]:
df = process_df(data)

In [11]:
df= df.cache() 

In [12]:
df.show()

+--------------------+-------------------+------+------------+----+-----------+--------+------------+
|         create_time|                 ts|job_id|custom_track| bid|campaign_id|group_id|publisher_id|
+--------------------+-------------------+------+------------+----+-----------+--------+------------+
|940c0aa0-0b5b-11e...|2022-07-24 14:19:07|  NULL|       alive|NULL|       NULL|    NULL|        NULL|
|d0435cd0-fea4-11e...|2022-07-08 10:00:36|  NULL|       click|NULL|       NULL|    NULL|        NULL|
|41bc7a20-0078-11e...|2022-07-10 17:46:41|  NULL|       click|NULL|       NULL|    NULL|        NULL|
|eb9bdd20-0379-11e...|2022-07-14 13:36:09|  NULL|  conversion|NULL|       NULL|    NULL|        NULL|
|f4a1e2e0-018d-11e...|2022-07-12 02:54:32|  NULL|       click|NULL|       NULL|    NULL|        NULL|
|8c575c80-0843-11e...|2022-07-20 15:49:32|  NULL|        NULL|NULL|       NULL|    NULL|        NULL|
|00b0fc30-ff9d-11e...|2022-07-09 15:37:12|  NULL|  conversion|NULL|       NULL|   

In [13]:
data = df 

In [14]:
#process click data 

click_data = data.filter(data.custom_track == 'click')
click_data.createOrReplaceTempView('clickdata')
click_data = spark.sql("""select date(ts) as date,hour(ts) as hour ,job_id,publisher_id,campaign_id,group_id,round(avg(bid),2) as bid_set , sum(bid) as spend_hour , count(*) as click from clickdata group by date(ts),
hour(ts),job_id,publisher_id,campaign_id,group_id""")

#process conversion data
conversion_data = data.filter(data.custom_track == 'conversion')
conversion_data.createOrReplaceTempView('conversiondata')
conversion_data = spark.sql("""select date(ts) as date,hour(ts) as hour ,job_id,publisher_id,campaign_id,group_id, count(*) as conversion from conversiondata group by date(ts),
hour(ts),job_id,publisher_id,campaign_id,group_id""")
#process qualified data
qualified_data = data.filter(data.custom_track == 'qualified')
qualified_data.createOrReplaceTempView('qualified')
qualified_data = spark.sql("""select date(ts) as date,hour(ts) as hour ,job_id,publisher_id,campaign_id,group_id, count(*) as qualified from qualified group by date(ts),
hour(ts),job_id,publisher_id,campaign_id,group_id""")
#process unqualified data 
unqualified_data = data.filter(data.custom_track == 'unqualified')
unqualified_data.createOrReplaceTempView('unqualified')
unqualified_data = spark.sql("""select date(ts) as date,hour(ts) as hour ,job_id,publisher_id,campaign_id,group_id, count(*) as unqualified from unqualified group by date(ts),
hour(ts),job_id,publisher_id,campaign_id,group_id""")
#filter null data 
click_data = click_data.filter(click_data.date.isNotNull())
conversion_data = conversion_data.filter(conversion_data.date.isNotNull())
qualified_data = qualified_data.filter(qualified_data.date.isNotNull())
unqualified_data = unqualified_data.filter(unqualified_data.date.isNotNull())

#finalize output full join
result = click_data.join(conversion_data, on = ['date','hour','job_id','publisher_id','campaign_id','group_id'],how='full').\
    join(qualified_data,on = ['date','hour','job_id','publisher_id','campaign_id','group_id'],how='full').\
        join(unqualified_data,on = ['date','hour','job_id','publisher_id','campaign_id','group_id'],how='full')

In [15]:
result.show()

+----------+----+------+------------+-----------+--------+-------+----------+-----+----------+---------+-----------+
|      date|hour|job_id|publisher_id|campaign_id|group_id|bid_set|spend_hour|click|conversion|qualified|unqualified|
+----------+----+------+------------+-----------+--------+-------+----------+-----+----------+---------+-----------+
|2022-07-06|   9|  NULL|        NULL|       NULL|    NULL|   NULL|      NULL|    1|      NULL|     NULL|       NULL|
|2022-07-06|  15|  NULL|        NULL|       NULL|    NULL|   NULL|      NULL| NULL|         2|     NULL|       NULL|
|2022-07-07|   2|  NULL|        NULL|       NULL|    NULL|   NULL|      NULL|    3|      NULL|     NULL|       NULL|
|2022-07-07|   3|  NULL|        NULL|       NULL|    NULL|   NULL|      NULL|    2|      NULL|     NULL|       NULL|
|2022-07-07|   3|   187|           1|         48|    NULL|   0.42|       2.5|    6|      NULL|     NULL|       NULL|
|2022-07-08|   2|  NULL|        NULL|       NULL|    NULL|   NUL

In [16]:
# 1. Sửa lại tên Database cho đúng với thực tế bạn đã tạo
url = "jdbc:mysql://localhost:3306/schema_name" 
driver = "com.mysql.cj.jdbc.Driver"
user = 'root'

# 2. Thay 'MẬT_KHẨU_DATAGRIP' bằng mật khẩu bạn dùng ở DataGrip (ví dụ: 'root' hoặc '123')
password = 'root' 

# 3. Câu SQL lấy dữ liệu từ bảng job
sql = "(SELECT id as job_id, company_id, group_id, campaign_id FROM job) A"

# 4. Thực hiện load
jobs = spark.read.format('jdbc') \
    .options(url=url, driver=driver, dbtable=sql, user=user, password=password) \
    .load()

# 5. Kiểm tra thử
jobs.show(5)

+------+----------+--------+-----------+
|job_id|company_id|group_id|campaign_id|
+------+----------+--------+-----------+
|     2|         1|      10|          1|
|     3|         1|      10|          1|
|     4|         1|      10|          1|
|     5|         1|      10|          1|
|     6|         1|      10|          1|
+------+----------+--------+-----------+
only showing top 5 rows



In [17]:
sql = '(SELECT * FROM events ) A'
events = spark.read.format('jdbc').options(url = url , driver = driver , dbtable = sql , user=user , password = password).load()

In [18]:
events.columns

['id',
 'job_id',
 'dates',
 'hours',
 'disqualified_application',
 'qualified_application',
 'conversion',
 'company_id',
 'group_id',
 'campaign_id',
 'publisher_id',
 'bid_set',
 'clicks',
 'impressions',
 'spend_hour',
 'sources']

In [19]:
jobs.show()

+------+----------+--------+-----------+
|job_id|company_id|group_id|campaign_id|
+------+----------+--------+-----------+
|     2|         1|      10|          1|
|     3|         1|      10|          1|
|     4|         1|      10|          1|
|     5|         1|      10|          1|
|     6|         1|      10|          1|
|     7|         1|      10|          1|
|     8|         1|      10|          1|
|     9|         1|      10|          1|
|    39|         1|    NULL|          1|
|    40|         1|      10|          1|
|    41|         1|      10|          1|
|    42|         1|      10|          1|
|    43|         1|    NULL|          1|
|    44|         1|      10|          1|
|    45|         1|      10|          1|
|    46|         1|      10|          1|
|    47|         1|      10|          1|
|    48|         1|      10|          1|
|    49|         1|      10|          1|
|    50|         1|      10|          1|
+------+----------+--------+-----------+
only showing top

In [20]:
final = result.join(jobs,on = 'job_id',how='left').drop(jobs.campaign_id).drop(jobs.group_id)

In [21]:
final = final.withColumn('updated_at',sf.lit(data.agg({'ts':'max'}).take(1)[0][0]))

In [22]:
final.show()

+------+----------+----+------------+-----------+--------+-------+----------+-----+----------+---------+-----------+----------+-------------------+
|job_id|      date|hour|publisher_id|campaign_id|group_id|bid_set|spend_hour|click|conversion|qualified|unqualified|company_id|         updated_at|
+------+----------+----+------------+-----------+--------+-------+----------+-----+----------+---------+-----------+----------+-------------------+
|  NULL|2022-07-06|   9|        NULL|       NULL|    NULL|   NULL|      NULL|    1|      NULL|     NULL|       NULL|      NULL|2022-07-27 15:58:35|
|  NULL|2022-07-06|  15|        NULL|       NULL|    NULL|   NULL|      NULL| NULL|         2|     NULL|       NULL|      NULL|2022-07-27 15:58:35|
|  NULL|2022-07-07|   2|        NULL|       NULL|    NULL|   NULL|      NULL|    3|      NULL|     NULL|       NULL|      NULL|2022-07-27 15:58:35|
|  NULL|2022-07-07|   3|        NULL|       NULL|    NULL|   NULL|      NULL|    2|      NULL|     NULL|       N

In [23]:
final = final.withColumnRenamed('date','dates')
final = final.withColumnRenamed('hour','hours')
final = final.withColumnRenamed('qualified','qualified_application')
final = final.withColumnRenamed('unqualified','disqualified_application')
final = final.withColumnRenamed('unqualified','disqualified_application')
final = final.withColumnRenamed('click','clicks')


In [24]:
final = final.withColumn('sources',sf.lit('Cassandra'))

In [25]:
final.columns

['job_id',
 'dates',
 'hours',
 'publisher_id',
 'campaign_id',
 'group_id',
 'bid_set',
 'spend_hour',
 'clicks',
 'conversion',
 'qualified_application',
 'disqualified_application',
 'company_id',
 'updated_at',
 'sources']

In [26]:
final.select(
 'job_id',
 'dates',
 'hours',
 'disqualified_application',
 'qualified_application',
 'conversion',
 'company_id',
 'group_id',
 'campaign_id',
 'publisher_id',
 'bid_set',
 'clicks',
 'spend_hour',
 'sources',
 'updated_at')

DataFrame[job_id: int, dates: date, hours: int, disqualified_application: bigint, qualified_application: bigint, conversion: bigint, company_id: string, group_id: int, campaign_id: int, publisher_id: int, bid_set: double, clicks: bigint, spend_hour: double, sources: string, updated_at: string]

In [27]:
# 1. Sửa 'schema_name' thành 'recruitment_dw'
mysql_url = "jdbc:mysql://localhost:3306/recruitment_dw" 
mysql_user = "root"

# 2. Kiểm tra lại mật khẩu: Nếu 'root' không được, hãy thử '1' hoặc '123' 
# (Mật khẩu bạn dùng để kết nối DataGrip thành công)
mysql_password = "root" 

final.write.format("jdbc") \
    .option("driver", "com.mysql.cj.jdbc.Driver") \
    .option("url", mysql_url) \
    .option("dbtable", "events") \
    .mode("append") \
    .option("user", mysql_user) \
    .option("password", mysql_password) \
    .save()

In [28]:
latest_time_cassandra = data.agg({'ts':'max'}).take(1)[0][0]

In [29]:
sql = '(select max(updated_at) as updated_at from events) A'
url = "jdbc:mysql://localhost:3306/recruitment_dw"
driver = "com.mysql.cj.jdbc.Driver"
user = 'root'
password = 'root'  
time_mysql = spark.read.format('jdbc').options(url = url , driver = driver , dbtable = sql , user=user , password = password).load()

In [30]:
import datetime

# Lấy giá trị ra trước
raw_value = time_mysql.take(1)[0][0]

if isinstance(raw_value, str):
    # Nếu đã là chuỗi, chỉ cần gán thẳng
    latest_time_sql = raw_value
else:
    # Nếu là đối tượng datetime, mới dùng strftime
    latest_time_sql = raw_value.strftime('%Y-%m-%d %H:%M:%S')

In [31]:
import datetime

In [32]:
latest_time_cassandra

'2022-07-27 15:58:35'

In [33]:
latest_time_sql

'2022-07-27 15:58:35'

In [34]:
import time

In [35]:
# Chạy thử cái này để check quyền và tên bảng
test_df = spark.read.format("jdbc") \
    .option("url", "jdbc:mysql://localhost:3306/recruitment_dw") \
    .option("dbtable", "events") \
    .option("user", "root") \
    .option("password", "root").load()
test_df.show(1)

+------+----------+-----+------------+-----------+--------+-------+----------+------+----------+---------------------+------------------------+----------+-------------------+---------+
|job_id|     dates|hours|publisher_id|campaign_id|group_id|bid_set|spend_hour|clicks|conversion|qualified_application|disqualified_application|company_id|         updated_at|  sources|
+------+----------+-----+------------+-----------+--------+-------+----------+------+----------+---------------------+------------------------+----------+-------------------+---------+
|  NULL|2022-07-06|    9|        NULL|       NULL|    NULL|   NULL|      NULL|     1|      NULL|                 NULL|                    NULL|      NULL|2022-07-27 15:58:35|Cassandra|
+------+----------+-----+------------+-----------+--------+-------+----------+------+----------+---------------------+------------------------+----------+-------------------+---------+
only showing top 1 row

